# 📋 Complete URAR Appraisal Report & Engagement Letter Generator V3
**Generates professional appraisal PDFs with matching engagement letters**

All shared fields (address, borrower, lender) are IDENTICAL between documents.

In [ ]:
# Cell 1: Install Dependencies
!pip install -q PyMuPDF pdf2image pytesseract opencv-python-headless reportlab Faker pandas pillow numpy
!apt-get install -qq tesseract-ocr poppler-utils
print("✅ Dependencies installed!")

In [ ]:
# Cell 2: Upload Reference PDF (optional - for learning structure)
from google.colab import files
print("📤 Upload reference appraisal PDF (optional, press Cancel to skip)...")
try:
    uploaded = files.upload()
    REF_PDF = list(uploaded.keys())[0] if uploaded else None
    print(f"✅ Reference: {REF_PDF}")
except:
    REF_PDF = None
    print("⏭️ Skipped reference upload")

In [ ]:
# Cell 3: Complete Data Model for URAR Appraisal
from faker import Faker
import random
from datetime import datetime, timedelta
from typing import Dict, List, Any
from dataclasses import dataclass, field, asdict

fake = Faker()

class AppraisalDataGenerator:
    """
    Generates complete URAR appraisal data with ALL fields.
    Data is shared between appraisal report and engagement letter.
    """
    
    STATES = ['FL', 'CA', 'TX', 'NY', 'AZ', 'CO', 'GA', 'NC', 'VA', 'WA']
    COUNTIES = {'FL': ['MIAMI-DADE', 'BROWARD', 'PALM BEACH', 'ORANGE', 'HILLSBOROUGH'],
                'CA': ['LOS ANGELES', 'SAN DIEGO', 'ORANGE', 'RIVERSIDE', 'SAN BERNARDINO'],
                'TX': ['HARRIS', 'DALLAS', 'TARRANT', 'BEXAR', 'TRAVIS']}
    LENDERS = ['A&D MORTGAGE LLC', 'QUICKEN LOANS', 'WELLS FARGO HOME MORTGAGE',
               'CHASE HOME LENDING', 'BANK OF AMERICA MORTGAGE', 'UNITED WHOLESALE MORTGAGE']
    APPRAISAL_COMPANIES = ['Sun City Appraisal, INC', 'Premier Valuations LLC',
                           'Accurate Appraisal Services', 'Elite Property Appraisers']
    
    def __init__(self):
        self.count = 0
    
    def generate(self) -> Dict:
        self.count += 1
        state = random.choice(self.STATES)
        county = random.choice(self.COUNTIES.get(state, [fake.city() + ' County']))
        
        # Property characteristics
        year_built = random.randint(1960, 2022)
        gla = random.randint(1500, 5000)
        site_acres = round(random.uniform(0.2, 6.0), 3)
        bedrooms = max(2, min(6, gla // 600))
        bathrooms = round(bedrooms * 0.75 + random.uniform(0.5, 1.5), 1)
        total_rooms = bedrooms + random.randint(3, 5)
        
        # Value calculations
        quality = random.choice(['Q2', 'Q3', 'Q4'])
        condition = random.choice(['C2', 'C3', 'C4'])
        q_mult = {'Q1': 1.5, 'Q2': 1.25, 'Q3': 1.0, 'Q4': 0.85}[quality]
        c_mult = {'C1': 1.15, 'C2': 1.05, 'C3': 0.95, 'C4': 0.85}[condition]
        base_psf = random.randint(250, 500)
        appraised_value = int(round(gla * base_psf * q_mult * c_mult, -3))
        
        # Dates
        inspection_date = datetime.now() - timedelta(days=random.randint(5, 20))
        effective_date = inspection_date
        report_date = inspection_date + timedelta(days=random.randint(5, 14))
        contract_date = inspection_date - timedelta(days=random.randint(15, 45))
        
        # Contract price (slightly different from appraised)
        contract_price = appraised_value + random.randint(-50000, 150000)
        
        # Generate addresses
        street_num = random.randint(1000, 19999)
        street_name = fake.street_name()
        street_suffix = random.choice(['St', 'Ave', 'Blvd', 'Dr', 'Ln', 'Ct', 'Way'])
        property_address = f"{street_num} {street_name} {street_suffix}"
        city = fake.city()
        zip_code = fake.zipcode()
        
        # Borrower/Owner
        borrower_first = fake.first_name().upper()
        borrower_last = fake.last_name().upper()
        borrower_name = f"{borrower_first} {borrower_last}"
        owner_company = f"{borrower_last.title()} Properties LLC"
        
        # Lender details
        lender = random.choice(self.LENDERS)
        lender_street = f"{random.randint(100, 9999)} {fake.street_name()} {random.choice(['Highway', 'Boulevard', 'Street'])}"
        lender_city = fake.city()
        lender_state = random.choice(self.STATES)
        lender_zip = fake.zipcode()
        
        # Appraiser details
        appraiser_first = fake.first_name()
        appraiser_last = fake.last_name()
        appraiser_name = f"{appraiser_first} {appraiser_last}"
        appraiser_company = random.choice(self.APPRAISAL_COMPANIES)
        appraiser_street = f"{random.randint(100, 9999)} {fake.street_name()} {random.choice(['Place', 'Avenue', 'Drive'])}"
        appraiser_city = city
        appraiser_phone = f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}"
        appraiser_email = f"{appraiser_first.lower()}.{appraiser_last.lower()}@appraisals.com"
        license_num = f"CERT.RES #R{random.choice(['D','G','A'])}{random.randint(1000,9999)}"
        license_exp = (datetime.now() + timedelta(days=random.randint(180, 730))).strftime('%m/%d/%Y')
        
        # Legal description
        section = random.randint(1, 36)
        township = random.randint(40, 60)
        range_val = random.randint(30, 50)
        legal_desc = f"{section} {township} {range_val} {site_acres} AC E1/2 OF NW1/4 OF NE1/4 OF NE1/4 LESS 40FT OR {random.randint(10000,99999)}-{random.randint(1000,9999)}"
        
        # Parcel and identifiers
        parcel = f"{random.randint(10,99)}-{random.randint(1000,9999)}-{random.randint(100,999)}-{random.randint(1000,9999)}"
        file_no = f"{random.randint(2300,2500)}-{random.randint(10000,99999)}"
        fha_case = str(random.randint(1000000, 9999999)) if random.random() > 0.7 else ""
        
        # Tax info
        tax_year = "2023"
        taxes = int(appraised_value * random.uniform(0.008, 0.015))
        
        # Cost approach
        site_value = int(appraised_value * random.uniform(0.25, 0.45))
        cost_psf = random.randint(200, 300)
        dwelling_cost = gla * cost_psf
        garage_sf = random.randint(400, 900)
        garage_cost = garage_sf * 60
        extras_cost = random.randint(50000, 200000)
        total_cost_new = dwelling_cost + garage_cost + extras_cost
        depreciation = int(total_cost_new * (datetime.now().year - year_built) * 0.012)
        depreciated_cost = total_cost_new - depreciation
        site_improvements = random.randint(30000, 80000)
        cost_value = site_value + depreciated_cost + site_improvements
        
        return {
            # ═══════════════════════════════════════════════════════
            # DOCUMENT META
            # ═══════════════════════════════════════════════════════
            'file_no': file_no,
            'fha_case_no': fha_case,
            
            # ═══════════════════════════════════════════════════════
            # SUBJECT PROPERTY - Complete Details
            # ═══════════════════════════════════════════════════════
            'property_address': property_address,
            'city': city,
            'state': state,
            'zip': zip_code,
            'county': county,
            'legal_description': legal_desc,
            'parcel_number': parcel,
            'tax_year': tax_year,
            'taxes': f"{taxes:,}",
            'special_assessments': "0",
            'hoa_fee': str(random.choice([0, 0, 0, 150, 250, 350])),
            'hoa_frequency': 'per month',
            'neighborhood_name': random.choice(['ACREAGE & UNREC', 'RESIDENTIAL ESTATES', 'COUNTRY CLUB', 'LAKEFRONT MANOR']),
            'map_reference': str(random.randint(30000, 40000)),
            'census_tract': f"{random.randint(100,999)}.{random.randint(10,99)}",
            'occupancy': random.choice(['Owner', 'Tenant', 'Vacant']),
            
            # Site details
            'site_area': f"{site_acres} ac",
            'site_area_sf': f"{int(site_acres * 43560):,}",
            'site_dimensions': random.choice(['NO SURVEY PROVIDED', f'{random.randint(100,300)} x {random.randint(200,600)}']),
            'site_shape': random.choice(['RECTANGULAR', 'IRREGULAR', 'SQUARE']),
            'view': 'N;Res;',
            'zoning': random.choice(['AU', 'R-1', 'R-2', 'RS-3', 'PUD']),
            'zoning_description': random.choice(['AGRICULTURE', 'SINGLE FAMILY', 'RESIDENTIAL', 'PLANNED UNIT DEV']),
            'zoning_compliance': 'Legal',
            'highest_best_use': 'Yes',
            
            # Utilities
            'utilities_electric': 'Public',
            'utilities_gas': random.choice(['Public', 'TANK', 'None']),
            'utilities_water': random.choice(['Public', 'WELL']),
            'utilities_sewer': random.choice(['Public', 'SEPTIC']),
            'street_type': random.choice(['PAVED ASPHALT', 'PAVED CONCRETE', 'GRAVEL']),
            'street_public': 'Public',
            'alley': 'NONE',
            
            # Flood
            'flood_zone': random.choice(['X', 'AE', 'AH', 'A']),
            'flood_map_number': f"12{random.randint(10,99)}C{random.randint(100,999)}L",
            'flood_map_date': f"{random.randint(1,12):02d}/{random.randint(1,28):02d}/{random.randint(2008,2022)}",
            'in_flood_zone': random.choice(['Yes', 'No']),
            
            # ═══════════════════════════════════════════════════════
            # BORROWER/OWNER - Complete Details
            # ═══════════════════════════════════════════════════════
            'borrower_name': borrower_name,
            'borrower_first_name': borrower_first,
            'borrower_last_name': borrower_last,
            'owner_of_record': owner_company,
            'borrower_phone': f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}",
            'borrower_email': f"{borrower_first.lower()}.{borrower_last.lower()}@email.com",
            
            # ═══════════════════════════════════════════════════════
            # LENDER/CLIENT - Complete Details
            # ═══════════════════════════════════════════════════════
            'lender_name': lender,
            'lender_address': f"{lender_street}, {lender_city}, {lender_state} {lender_zip}",
            'lender_street': lender_street,
            'lender_city': lender_city,
            'lender_state': lender_state,
            'lender_zip': lender_zip,
            'lender_contact': fake.name(),
            'lender_phone': f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}",
            'lender_email': f"orders@{lender.lower().replace(' ', '').replace(',', '')[:15]}.com",
            'amc_name': random.choice(['FASTAPP APPRAISAL MANAGEMENT COMPANY', 'COESTER VMS', 'MERCURY NETWORK', '']),
            
            # ═══════════════════════════════════════════════════════
            # APPRAISER - Complete Details
            # ═══════════════════════════════════════════════════════
            'appraiser_name': appraiser_name,
            'appraiser_company': appraiser_company,
            'appraiser_address': f"{appraiser_street}, {appraiser_city}, {state} {zip_code}",
            'appraiser_street': appraiser_street,
            'appraiser_city': appraiser_city,
            'appraiser_state': state,
            'appraiser_zip': zip_code,
            'appraiser_phone': appraiser_phone,
            'appraiser_email': appraiser_email,
            'license_number': license_num,
            'license_state': state,
            'license_expiry': license_exp,
            
            # ═══════════════════════════════════════════════════════
            # CONTRACT/SALE - Complete Details
            # ═══════════════════════════════════════════════════════
            'assignment_type': random.choice(['Purchase Transaction', 'Refinance Transaction']),
            'property_rights': 'Fee Simple',
            'contract_price': f"{contract_price:,}",
            'contract_price_raw': contract_price,
            'contract_date': contract_date.strftime('%m/%d/%Y'),
            'is_seller_owner': random.choice(['Yes', 'No']),
            'is_offered_for_sale': random.choice(['Yes', 'No']),
            'offering_price': f"{contract_price:,}" if random.random() > 0.5 else "",
            'dom': str(random.randint(0, 180)),
            'financing_type': random.choice(['Conv;0', 'FHA;0', 'VA;0', 'Cash;0']),
            'concessions': '$0;;',
            'concessions_amount': '0',
            'arms_length': 'ArmLth',
            
            # ═══════════════════════════════════════════════════════
            # DATES
            # ═══════════════════════════════════════════════════════
            'inspection_date': inspection_date.strftime('%m/%d/%Y'),
            'effective_date': effective_date.strftime('%m/%d/%Y'),
            'report_date': report_date.strftime('%m/%d/%Y'),
            'order_date': (inspection_date - timedelta(days=random.randint(3, 7))).strftime('%m/%d/%Y'),
            'inspection_due': (inspection_date + timedelta(days=1)).strftime('%m/%d/%Y'),
            'report_due': report_date.strftime('%m/%d/%Y'),
            
            # ═══════════════════════════════════════════════════════
            # NEIGHBORHOOD
            # ═══════════════════════════════════════════════════════
            'location_type': random.choice(['Urban', 'Suburban', 'Rural']),
            'built_up': random.choice(['Over 75%', '25-75%', 'Under 25%']),
            'growth': random.choice(['Rapid', 'Stable', 'Slow']),
            'property_values': random.choice(['Increasing', 'Stable', 'Declining']),
            'demand_supply': random.choice(['Shortage', 'In Balance', 'Over Supply']),
            'marketing_time': random.choice(['Under 3 mths', '3-6 mths', 'Over 6 mths']),
            'price_low': f"{random.randint(150, 400)},000",
            'price_high': f"{random.randint(1800, 3000)},000",
            'price_pred': f"{random.randint(600, 1200)},000",
            'age_low': str(random.randint(1, 5)),
            'age_high': str(random.randint(80, 100)),
            'age_pred': str(random.randint(30, 50)),
            'land_use_1unit': str(random.randint(55, 75)),
            'land_use_2to4': str(random.randint(0, 10)),
            'land_use_multi': str(random.randint(0, 10)),
            'land_use_comm': str(random.randint(5, 15)),
            'land_use_other': str(random.randint(10, 25)),
            'neighborhood_boundaries': f"NORTH OF SW {random.randint(200,280)}TH STREET, SOUTH OF SW {random.randint(100,150)}TH STREET, EAST OF SW {random.randint(180,230)}TH AVENUE AND WEST OF KROME AVENUE.",
            'neighborhood_desc': "THE SUBJECT IS LOCATED IN AN ESTABLISHED RESIDENTIAL NEIGHBORHOOD. IT IS WITHIN A REASONABLE DISTANCE TO ALL AREA AMENITIES WITH ADEQUATE ACCESS TO MAJOR ARTERIES OF TRANSPORTATION AND PLACES OF EMPLOYMENT.",
            'market_conditions': "STATISTICS INDICATE THAT PROPERTY VALUES ARE STABILIZING WITH A NOTE IN BALANCE. FINANCING IS READILY AVAILABLE.",
            
            # ═══════════════════════════════════════════════════════
            # IMPROVEMENTS
            # ═══════════════════════════════════════════════════════
            'num_units': 'One',
            'num_stories': str(random.choice([1, 1, 2])),
            'unit_type': 'Det.',
            'design_style': random.choice(['RAMBLER', 'COLONIAL', 'CONTEMPORARY', 'RANCH', 'SPLIT LEVEL']),
            'existing_proposed': 'Existing',
            'year_built': str(year_built),
            'effective_age': str(max(5, datetime.now().year - year_built - random.randint(5, 20))),
            
            # Foundation
            'foundation_type': random.choice(['Concrete Slab', 'Crawl Space', 'Full Basement']),
            'foundation_walls': 'CONCRETE/AVG',
            'basement_area': '0',
            'basement_finish': '0%',
            
            # Exterior
            'exterior_walls': random.choice(['CBS/AVERAGE', 'BRICK/AVERAGE', 'FRAME/AVERAGE', 'STUCCO/AVERAGE']),
            'roof_surface': random.choice(['TILE/AVE', 'SHINGLE/AVG', 'METAL/AVG']),
            'gutters': random.choice(['YES/AVG', 'NO/NO']),
            'window_type': random.choice(['SLIDING.IMP/GOOD', 'DOUBLE HUNG/AVG', 'CASEMENT/AVG']),
            'storm_sash': 'NONE',
            'screens': 'YES/AVG',
            
            # Interior
            'floors_material': 'TILE/AVG',
            'walls_material': 'DRYWALL/AVG',
            'trim_finish': 'WOOD/AVG',
            'bath_floor': 'TILE/AVG',
            'bath_wainscot': 'TILE/AVG',
            
            # Systems
            'heating_type': random.choice(['FWA', 'HWBB', 'Radiant']),
            'heating_fuel': 'ELECTRIC',
            'cooling': 'Central Air Conditioning',
            
            # Rooms
            'total_rooms': str(total_rooms),
            'bedrooms': str(bedrooms),
            'bathrooms': str(bathrooms),
            'gla': f"{gla:,}",
            'gla_raw': gla,
            
            # Amenities
            'fireplace_count': str(random.choice([0, 0, 1, 2])),
            'patio_deck': random.choice(['PAV', 'WOOD DECK', 'SCREENED']),
            'porch': random.choice(['COV.ENT', 'OPEN', 'SCREENED']),
            'pool': random.choice(['N-GROUND', 'ABOVE', 'NONE']),
            'fence': random.choice(['CHN LINK', 'WOOD', 'VINYL', 'NONE']),
            'other_amenities': random.choice(['GUEST H', 'WORKSHOP', 'BARN', '']),
            
            # Garage
            'driveway_cars': str(random.randint(4, 10)),
            'driveway_surface': random.choice(['PAVERS', 'CONCRETE', 'ASPHALT']),
            'garage_cars': str(random.randint(1, 3)),
            'garage_type': random.choice(['Att.', 'Det.', 'Built-in']),
            'carport_cars': str(random.choice([0, 0, 2])),
            'garage_sf': garage_sf,
            
            # Appliances
            'has_refrigerator': True,
            'has_range': True,
            'has_dishwasher': True,
            'has_disposal': True,
            'has_microwave': True,
            'has_washer_dryer': True,
            
            # Quality/Condition
            'quality': quality,
            'condition': condition,
            'condition_desc': f"{condition};Kitchen-remodeled-one to five years ago;Bathrooms-remodeled-one to five years ago;AT THE TIME OF THE INSPECTION, THE SUBJECT PROPERTY WAS CONSIDERED TO BE IN AVERAGE CONDITION.",
            'additional_features': 'WOOD CABINETS, QUARTZ COUNTERTOPS, IMPACT DOORS AND WINDOWS.',
            
            # ═══════════════════════════════════════════════════════
            # VALUES
            # ═══════════════════════════════════════════════════════
            'appraised_value': f"{appraised_value:,}",
            'appraised_value_raw': appraised_value,
            'sales_comp_value': f"{appraised_value:,}",
            'cost_approach_value': f"{cost_value:,}",
            'site_value': f"{site_value:,}",
            'dwelling_cost': f"{dwelling_cost:,}",
            'dwelling_cost_psf': cost_psf,
            'garage_cost': f"{garage_cost:,}",
            'extras_cost': f"{extras_cost:,}",
            'total_cost_new': f"{total_cost_new:,}",
            'depreciation': f"{depreciation:,}",
            'depreciated_cost': f"{depreciated_cost:,}",
            'site_improvements': f"{site_improvements:,}",
            'remaining_economic_life': str(random.randint(40, 60)),
            
            # ═══════════════════════════════════════════════════════
            # PRIOR SALES
            # ═══════════════════════════════════════════════════════
            'subject_prior_date': (datetime.now() - timedelta(days=random.randint(400, 1000))).strftime('%m/%d/%Y'),
            'subject_prior_price': f"{int(appraised_value * random.uniform(0.5, 0.75)):,}",
            
            # ═══════════════════════════════════════════════════════
            # ENGAGEMENT LETTER SPECIFIC
            # ═══════════════════════════════════════════════════════
            'order_number': f"ES-{random.randint(100000, 999999)}",
            'loan_number': str(random.randint(100000000, 999999999)),
            'appraisal_fee': random.choice([475, 500, 525, 550, 600, 650]),
            'rush_fee': random.choice([0, 0, 0, 75, 100, 150]),
            'report_type_letter': random.choice(['URAR', 'FHA', 'VA']),
            'client_name': 'Equity Solutions Appraisal Services',
            'client_email': 'orders@equitysolutions.com',
            'client_phone': f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}",
            
            # ═══════════════════════════════════════════════════════
            # COMPARABLES (will be generated separately)
            # ═══════════════════════════════════════════════════════
            'comparables': self._generate_comparables(appraised_value, gla, site_acres, city, state),
        }
    
    def _generate_comparables(self, subject_value, subject_gla, subject_site, city, state):
        comps = []
        directions = ['N', 'S', 'E', 'W', 'NW', 'NE', 'SW', 'SE']
        
        for i in range(6):
            distance = round(random.uniform(0.05, 6.0), 2)
            direction = random.choice(directions)
            
            comp_gla = subject_gla + random.randint(-800, 1200)
            comp_site = round(subject_site + random.uniform(-2, 2), 2)
            comp_sale_price = subject_value + random.randint(-300000, 400000)
            comp_beds = max(2, min(6, comp_gla // 600))
            comp_baths = round(comp_beds * 0.75 + random.uniform(0, 1), 1)
            comp_age = random.randint(25, 40)
            
            # Adjustments
            gla_adj = int((subject_gla - comp_gla) * random.randint(80, 120))
            site_adj = int((subject_site - comp_site) * random.randint(15000, 25000))
            
            comps.append({
                'id': i + 1,
                'address': f"{random.randint(10000, 25000)} SW {random.randint(150, 280)}th {random.choice(['St', 'Ave', 'Ct'])}",
                'city': city,
                'state': state,
                'zip': fake.zipcode(),
                'proximity': f"{distance} miles {direction}",
                'sale_price': comp_sale_price,
                'sale_price_formatted': f"{comp_sale_price:,}",
                'price_per_sf': round(comp_sale_price / comp_gla, 2),
                'sale_date': f"s{random.randint(1,12):02d}/{random.randint(22,23)};c{random.randint(1,12):02d}/{random.randint(22,23)}",
                'mls_number': f"A{random.randint(10000000, 99999999)}",
                'dom': random.randint(30, 200),
                'financing': random.choice(['Conv;0', 'Cash;0', 'FHA;0']),
                'concessions': 'ArmLth',
                'site': f"{comp_site} ac",
                'site_adj': site_adj,
                'view': 'N;Res;',
                'design': f"DT{random.choice([1,2])};RAMBLER",
                'quality': random.choice(['Q2', 'Q3', 'Q4']),
                'condition': random.choice(['C2', 'C3', 'C4']),
                'age': comp_age,
                'gla': comp_gla,
                'gla_adj': gla_adj,
                'total_rooms': comp_beds + random.randint(3, 5),
                'bedrooms': comp_beds,
                'bathrooms': comp_baths,
                'room_adj': random.randint(-10000, 10000),
                'basement': '0sf',
                'functional': 'AVERAGE',
                'heating_cooling': 'CENTRAL',
                'energy': 'HIGH EFFIC',
                'garage_carport': f"{random.randint(1,2)}ga{random.randint(0,2)}cp{random.randint(4,10)}dw",
                'garage_adj': random.randint(0, 25000),
                'porch_patio': 'COV ENTRY',
                'fence_pool': random.choice(['FENCE/POOL', 'FENCED', 'NONE']),
                'pool_adj': random.randint(0, 50000),
                'guest_house': random.choice(['NONE', 'GST HSE', 'WORKSHOP']),
                'guest_adj': random.randint(0, 50000),
                'net_adjustment': 0,  # Will calculate
                'gross_adjustment': 0,  # Will calculate
                'adjusted_price': 0,  # Will calculate
            })
            
            # Calculate totals
            c = comps[-1]
            net = c['site_adj'] + c['gla_adj'] + c['room_adj'] + c['garage_adj'] + c['pool_adj'] + c['guest_adj']
            gross = abs(c['site_adj']) + abs(c['gla_adj']) + abs(c['room_adj']) + abs(c['garage_adj']) + abs(c['pool_adj']) + abs(c['guest_adj'])
            c['net_adjustment'] = net
            c['gross_adjustment'] = gross
            c['adjusted_price'] = c['sale_price'] + net
            c['net_adj_pct'] = round(abs(net) / c['sale_price'] * 100, 1)
            c['gross_adj_pct'] = round(gross / c['sale_price'] * 100, 1)
        
        return comps

# Test
gen = AppraisalDataGenerator()
sample = gen.generate()
print("✅ Data Generator Ready!")
print(f"   📍 {sample['property_address']}, {sample['city']}, {sample['state']} {sample['zip']}")
print(f"   👤 Borrower: {sample['borrower_name']}")
print(f"   🏦 Lender: {sample['lender_name']}")
print(f"   📋 Appraiser: {sample['appraiser_name']}, {sample['appraiser_company']}")
print(f"   💰 Value: ${sample['appraised_value']} | GLA: {sample['gla']} SF | {sample['bedrooms']} BR/{sample['bathrooms']} BA")
print(f"   📊 {len(sample['comparables'])} Comparables generated")

In [ ]:
# Cell 4: Cover Page & URAR Generatorsfrom reportlab.pdfgen import canvasfrom reportlab.lib.pagesizes import letterfrom reportlab.lib.colors import black, white, lightgreyfrom reportlab.lib.units import inchclass CoverPageGenerator:    """Generates professional appraisal cover page"""        def __init__(self):        self.page_w, self.page_h = letter        def generate(self, c, data):        y = self.page_h - 1*inch                # Title        c.setFont('Helvetica-Bold', 14)        c.drawCentredString(self.page_w/2, y, 'APPRAISAL OF REAL PROPERTY')        y -= 25        c.setFont('Helvetica', 10)        c.drawCentredString(self.page_w/2, y, 'LOCATED AT')        y -= 20                # Property Address        c.setFont('Helvetica-Bold', 12)        c.drawCentredString(self.page_w/2, y, data['property_address'])        y -= 16        c.drawCentredString(self.page_w/2, y, f"{data['city']}, {data['state']} {data['zip']}")        y -= 25                # Legal description        c.setFont('Helvetica', 7)        legal = data.get('legal_description', '')[:100]        c.drawCentredString(self.page_w/2, y, legal)        y -= 35                # FOR section        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'FOR')        y -= 18        c.setFont('Helvetica', 11)        c.drawCentredString(self.page_w/2, y, data['lender_name'])        y -= 14        c.setFont('Helvetica', 9)        c.drawCentredString(self.page_w/2, y, data['lender_address'])        y -= 35                # AS OF        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'AS OF')        y -= 18        c.setFont('Helvetica', 11)        c.drawCentredString(self.page_w/2, y, data['effective_date'])        y -= 35                # BY section        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'BY')        y -= 18        c.setFont('Helvetica', 11)        c.drawCentredString(self.page_w/2, y, data['appraiser_name'])        y -= 14        c.drawCentredString(self.page_w/2, y, data['appraiser_company'])        y -= 14        c.setFont('Helvetica', 9)        c.drawCentredString(self.page_w/2, y, data['appraiser_address'])        y -= 12        c.drawCentredString(self.page_w/2, y, data['appraiser_phone'])        y -= 12        c.drawCentredString(self.page_w/2, y, data['appraiser_email'])                # Header box at bottom        c.setFont('Helvetica', 7)        box_y = 2.2*inch        c.drawString(0.5*inch, box_y, f"FHA/VA Case No. {data.get('fha_case_no', '')}")        c.drawString(0.5*inch, box_y-12, f"Borrower: {data['borrower_name']}")        c.drawString(0.5*inch, box_y-24, f"File No. {data['file_no']}")        c.drawString(3.5*inch, box_y, f"Property Address: {data['property_address']}")        c.drawString(3.5*inch, box_y-12, f"City: {data['city']}  County: {data['county']}  State: {data['state']}  Zip: {data['zip']}")        c.drawString(3.5*inch, box_y-24, f"Lender/Client: {data['lender_name']}")                # Footer        c.setFont('Helvetica', 6)        c.drawCentredString(self.page_w/2, 0.4*inch, 'Form GA1NV - "TOTAL" appraisal software by a la mode, inc. - 1-800-ALAMODE')class URARPageGenerator:    """Generates URAR form pages"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 0.3*inch        self.content_w = self.page_w - 2*self.margin        def _header(self, c, data, page_num):        y = self.page_h - 0.25*inch        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, f"FHA/VA Case No. {data.get('fha_case_no', '')}")        c.drawRightString(self.page_w - self.margin, y, f"File # {data['file_no']}")        c.setFont('Helvetica-Bold', 8)        c.drawCentredString(self.page_w/2, y, 'Uniform Residential Appraisal Report')        return y - 12        def _section_header(self, c, y, title):        c.setFillColor(lightgrey)        c.rect(self.margin, y-12, self.content_w, 12, fill=True, stroke=True)        c.setFillColor(black)        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin+3, y-10, title)        return y - 12        def _footer(self, c, page_num):        c.setFont('Helvetica', 5)        c.drawString(self.margin, 0.3*inch, 'Freddie Mac Form 70  March 2005')        c.drawRightString(self.page_w - self.margin, 0.3*inch, 'Fannie Mae Form 1004  March 2005')        c.drawCentredString(self.page_w/2, 0.3*inch, f'Page {page_num} of 6')        def generate_page1(self, c, data):        """URAR Page 1 - Subject, Neighborhood, Site, Improvements"""        y = self._header(c, data, 1)                # SUBJECT Section        y = self._section_header(c, y, 'SUBJECT')        c.setFont('Helvetica', 6)        y -= 10        c.drawString(self.margin, y, f"Property Address: {data['property_address']}   City: {data['city']}   State: {data['state']}   Zip: {data['zip']}")        y -= 10        c.drawString(self.margin, y, f"Borrower: {data['borrower_name']}   Owner of Record: {data['owner_of_record']}   County: {data['county']}")        y -= 10        c.drawString(self.margin, y, f"Legal Description: {data['legal_description'][:70]}")        y -= 10        c.drawString(self.margin, y, f"Assessor's Parcel #: {data['parcel_number']}   Tax Year: {data['tax_year']}   R.E. Taxes $: {data['taxes']}")        y -= 10        c.drawString(self.margin, y, f"Neighborhood: {data['neighborhood_name']}   Map Ref: {data['map_reference']}   Census Tract: {data['census_tract']}")        y -= 10        c.drawString(self.margin, y, f"Occupant: {data['occupancy']}   Special Assessments $: {data['special_assessments']}   HOA $: {data['hoa_fee']} {data['hoa_frequency']}")        y -= 12        c.drawString(self.margin, y, f"Property Rights: {data['property_rights']}   Assignment Type: {data['assignment_type']}")        y -= 10        c.drawString(self.margin, y, f"Lender/Client: {data['lender_name']}   Address: {data['lender_address'][:50]}")        y -= 15                if data['assignment_type'] == 'Purchase Transaction':            c.drawString(self.margin, y, f"Contract Price $: {data['contract_price']}   Date: {data['contract_date']}   Is seller owner? {data['is_seller_owner']}")            y -= 15                # NEIGHBORHOOD Section        y = self._section_header(c, y, 'NEIGHBORHOOD')        y -= 10        c.drawString(self.margin, y, f"Location: {data['location_type']}   Built-Up: {data['built_up']}   Growth: {data['growth']}   Property Values: {data['property_values']}")        y -= 10        c.drawString(self.margin, y, f"Demand/Supply: {data['demand_supply']}   Marketing Time: {data['marketing_time']}")        y -= 10        c.drawString(self.margin, y, f"Price Range: ${data['price_low']} - ${data['price_high']}   Predominant: ${data['price_pred']}")        y -= 10        c.drawString(self.margin, y, f"Age Range: {data['age_low']} - {data['age_high']} yrs   Predominant: {data['age_pred']} yrs")        y -= 10        c.drawString(self.margin, y, f"Land Use: 1-Unit {data['land_use_1unit']}%  2-4 Unit {data['land_use_2to4']}%  Multi {data['land_use_multi']}%  Comm {data['land_use_comm']}%  Other {data['land_use_other']}%")        y -= 10        c.drawString(self.margin, y, f"Boundaries: {data['neighborhood_boundaries'][:80]}")        y -= 10        c.drawString(self.margin, y, f"Description: {data['neighborhood_desc'][:90]}")        y -= 15                # SITE Section        y = self._section_header(c, y, 'SITE')        y -= 10        c.drawString(self.margin, y, f"Dimensions: {data['site_dimensions']}   Area: {data['site_area']}   Shape: {data['site_shape']}   View: {data['view']}")        y -= 10        c.drawString(self.margin, y, f"Zoning: {data['zoning']} - {data['zoning_description']}   Compliance: {data['zoning_compliance']}   Highest & Best Use: {data['highest_best_use']}")        y -= 10        c.drawString(self.margin, y, f"Utilities - Electric: {data['utilities_electric']}  Gas: {data['utilities_gas']}  Water: {data['utilities_water']}  Sewer: {data['utilities_sewer']}")        y -= 10        c.drawString(self.margin, y, f"Street: {data['street_type']}   Alley: {data['alley']}")        y -= 10        c.drawString(self.margin, y, f"FEMA Flood Zone: {data['flood_zone']}   Map #: {data['flood_map_number']}   Date: {data['flood_map_date']}")        y -= 15                # IMPROVEMENTS Section        y = self._section_header(c, y, 'IMPROVEMENTS')        y -= 10        c.drawString(self.margin, y, f"Units: {data['num_units']}   Stories: {data['num_stories']}   Type: {data['unit_type']}   Design: {data['design_style']}   Year Built: {data['year_built']}   Eff Age: {data['effective_age']} yrs")        y -= 10        c.drawString(self.margin, y, f"Foundation: {data['foundation_type']}   Basement: {data['basement_area']} sf   Finish: {data['basement_finish']}")        y -= 10        c.drawString(self.margin, y, f"Exterior: {data['exterior_walls']}   Roof: {data['roof_surface']}   Gutters: {data['gutters']}   Windows: {data['window_type']}")        y -= 10        c.drawString(self.margin, y, f"Interior - Floors: {data['floors_material']}   Walls: {data['walls_material']}   Trim: {data['trim_finish']}   Bath: {data['bath_floor']}")        y -= 10        c.drawString(self.margin, y, f"Heating: {data['heating_type']} / {data['heating_fuel']}   Cooling: {data['cooling']}")        y -= 10        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin, y, f"Finished Area: {data['total_rooms']} Rooms   {data['bedrooms']} Bedrooms   {data['bathrooms']} Bath(s)   {data['gla']} Sq.Ft. GLA")        y -= 10        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, f"Amenities - Fireplace: {data['fireplace_count']}   Patio: {data['patio_deck']}   Porch: {data['porch']}   Pool: {data['pool']}   Fence: {data['fence']}   Other: {data['other_amenities']}")        y -= 10        c.drawString(self.margin, y, f"Garage: {data['garage_cars']} cars {data['garage_type']}   Carport: {data['carport_cars']} cars   Driveway: {data['driveway_cars']} cars {data['driveway_surface']}")        y -= 10        c.drawString(self.margin, y, f"Quality: {data['quality']}   Condition: {data['condition']}   Features: {data['additional_features'][:60]}")                self._footer(c, 1)print("✅ Cover Page & URAR Generators ready!")

In [ ]:
# Cell 5: Sales Comparison & Engagement Letter Generatorsclass SalesComparisonGenerator:    """Generates Sales Comparison Approach pages"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 0.3*inch        def generate_page2(self, c, data):        """URAR Page 2 - Sales Comparison Grid"""        y = self.page_h - 0.25*inch        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, f"FHA/VA Case No. {data.get('fha_case_no', '')}")        c.drawRightString(self.page_w - self.margin, y, f"File # {data['file_no']}")        c.setFont('Helvetica-Bold', 8)        c.drawCentredString(self.page_w/2, y, 'Uniform Residential Appraisal Report')        y -= 15                # Sales Comparison Header        c.setFillColor(lightgrey)        c.rect(self.margin, y-12, self.page_w - 2*self.margin, 12, fill=True, stroke=True)        c.setFillColor(black)        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin+3, y-10, 'SALES COMPARISON APPROACH')        y -= 15                # Comparable grid header        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, f"Comparable sales range: ${data.get('price_low', '1,500,000')} to ${data.get('price_high', '2,300,000')}")        y -= 12                # Column headers        col_w = (self.page_w - 2*self.margin) / 4        headers = ['SUBJECT', 'COMPARABLE 1', 'COMPARABLE 2', 'COMPARABLE 3']        c.setFont('Helvetica-Bold', 6)        for i, h in enumerate(headers):            c.drawString(self.margin + i*col_w + 5, y, h)        y -= 10                # Address row        c.setFont('Helvetica', 5)        comps = data['comparables'][:3]        c.drawString(self.margin + 5, y, data['property_address'][:25])        for i, comp in enumerate(comps):            c.drawString(self.margin + (i+1)*col_w + 5, y, comp['address'][:25])        y -= 8                # City row        c.drawString(self.margin + 5, y, f"{data['city']}, {data['state']}")        for i, comp in enumerate(comps):            c.drawString(self.margin + (i+1)*col_w + 5, y, f"{comp['city']}, {comp['state']}")        y -= 10                # Proximity        c.drawString(self.margin + 5, y, "---")        for i, comp in enumerate(comps):            c.drawString(self.margin + (i+1)*col_w + 5, y, comp['proximity'])        y -= 10                # Sale Price        c.setFont('Helvetica-Bold', 6)        c.drawString(self.margin + 5, y, f"${data['contract_price']}")        for i, comp in enumerate(comps):            c.drawString(self.margin + (i+1)*col_w + 5, y, f"${comp['sale_price_formatted']}")        y -= 8                # Price/SF        c.setFont('Helvetica', 5)        c.drawString(self.margin + 5, y, f"${round(data['appraised_value_raw']/data['gla_raw'], 2)}/sf")        for i, comp in enumerate(comps):            c.drawString(self.margin + (i+1)*col_w + 5, y, f"${comp['price_per_sf']}/sf")        y -= 10                # Data source        for i, comp in enumerate(comps):            c.drawString(self.margin + (i+1)*col_w + 5, y, f"MLS #{comp['mls_number']}")        y -= 15                # Adjustment rows        adj_fields = [            ('Sale/Financing', 'arms_length', 'financing'),            ('Date of Sale', 'contract_date', 'sale_date'),            ('Site', 'site_area', 'site'),            ('View', 'view', 'view'),            ('Design', 'design_style', 'design'),            ('Quality', 'quality', 'quality'),            ('Condition', 'condition', 'condition'),            ('GLA', 'gla', 'gla'),            ('Room Count', 'total_rooms', 'total_rooms'),        ]                for label, subj_key, comp_key in adj_fields:            c.setFont('Helvetica', 5)            c.drawString(self.margin, y, label)            c.drawString(self.margin + 5, y-7, str(data.get(subj_key, ''))[:15])            for i, comp in enumerate(comps):                val = str(comp.get(comp_key, ''))[:15]                c.drawString(self.margin + (i+1)*col_w + 5, y-7, val)            y -= 14                # Net/Gross adjustments        y -= 5        c.setFont('Helvetica-Bold', 6)        c.drawString(self.margin, y, "Net Adjustment")        for i, comp in enumerate(comps):            adj = comp['net_adjustment']            sign = '+' if adj >= 0 else ''            c.drawString(self.margin + (i+1)*col_w + 5, y, f"{sign}${adj:,}")        y -= 10                c.drawString(self.margin, y, "Adjusted Price")        for i, comp in enumerate(comps):            c.drawString(self.margin + (i+1)*col_w + 5, y, f"${comp['adjusted_price']:,}")        y -= 20                # Reconciliation        c.setFillColor(lightgrey)        c.rect(self.margin, y-12, self.page_w - 2*self.margin, 12, fill=True, stroke=True)        c.setFillColor(black)        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin+3, y-10, 'RECONCILIATION')        y -= 20                c.setFont('Helvetica', 6)        c.drawString(self.margin, y, f"Indicated Value by Sales Comparison Approach: ${data['sales_comp_value']}")        y -= 12        c.drawString(self.margin, y, f"Indicated Value by Cost Approach: ${data['cost_approach_value']}")        y -= 20                c.setFont('Helvetica-Bold', 8)        c.drawString(self.margin, y, f"APPRAISED VALUE: ${data['appraised_value']}   as of {data['effective_date']}")                # Footer        c.setFont('Helvetica', 5)        c.drawString(self.margin, 0.3*inch, 'Freddie Mac Form 70  March 2005')        c.drawRightString(self.page_w - self.margin, 0.3*inch, 'Fannie Mae Form 1004  March 2005')        c.drawCentredString(self.page_w/2, 0.3*inch, 'Page 2 of 6')class EngagementLetterGenerator:    """Generates professional engagement letters with MATCHING data"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 1*inch        def generate(self, c, data):        y = self.page_h - 1*inch                # Header        c.setFont('Helvetica-Bold', 12)        c.drawString(self.margin, y, data['client_name'].upper())        y -= 18        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Appraisal Engagement / Order Confirmation')        y -= 25                # Order info        c.setFont('Helvetica', 9)        c.drawString(self.margin, y, f"Order Date: {data['order_date']}                    Order Number: {data['order_number']}")        y -= 12        c.drawString(self.margin, y, f"Loan Number: {data['loan_number']}")        y -= 25                # Client Info Section        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Client Information')        y -= 15        c.setFont('Helvetica', 9)        c.drawString(self.margin, y, f"Client: {data['client_name']}")        y -= 12        c.drawString(self.margin, y, f"Email: {data['client_email']}")        y -= 12        c.drawString(self.margin, y, f"Phone: {data['client_phone']}")        y -= 12        c.drawString(self.margin, y, f"Lender: {data['lender_name']}")        y -= 12        c.drawString(self.margin, y, f"Borrower: {data['borrower_name']}")        y -= 25                # Subject Property Section (MATCHING data)        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Subject Property')        y -= 15                # Draw box around property        box_h = 55        c.rect(self.margin, y - box_h, self.page_w - 2*self.margin, box_h)                c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin + 10, y - 15, data['property_address'])        c.setFont('Helvetica', 9)        c.drawString(self.margin + 10, y - 28, f"{data['city']}, {data['state']} {data['zip']}")        c.drawString(self.margin + 10, y - 41, f"Property Type: Single Family Residential")        c.drawString(self.margin + 10, y - 54, f"Occupancy: {data['occupancy']}")        y -= box_h + 20                # Purpose        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Purpose of Appraisal')        y -= 15        c.setFont('Helvetica', 9)        c.drawString(self.margin, y, 'To develop an opinion of market value.')        y -= 12        c.drawString(self.margin, y, 'Intended Use: Mortgage loan underwriting.')        y -= 12        c.drawString(self.margin, y, f"Intended User: {data['client_name']} and the lender/client.")        y -= 25                # Report Type        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Report Type')        y -= 15        c.setFont('Helvetica', 9)        types = ['URAR - Form 1004', 'FHA', 'VA', 'Other']        for t in types:            check = '☑' if data['report_type_letter'] in t else '☐'            c.drawString(self.margin, y, f"{check} {t}")            y -= 12        y -= 15                # Turn Time        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Turn Time')        y -= 15        c.setFont('Helvetica', 9)        c.drawString(self.margin, y, f"Inspection Due: {data['inspection_due']}")        y -= 12        c.drawString(self.margin, y, f"Report Due: {data['report_due']}")        y -= 25                # Fee        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Fee & Payment')        y -= 15        c.setFont('Helvetica', 9)        c.drawString(self.margin, y, f"Appraisal Fee: ${data['appraisal_fee']:.2f}")        y -= 12        if data['rush_fee'] > 0:            c.drawString(self.margin, y, f"Rush Fee: ${data['rush_fee']:.2f}")            y -= 12        c.drawString(self.margin, y, "Payment Terms: As agreed per vendor agreement.")        y -= 25                # Confidentiality        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Confidentiality')        y -= 15        c.setFont('Helvetica', 8)        c.drawString(self.margin, y, "This appraisal report is confidential and intended solely for the client and lender identified.")        y -= 25                # Acceptance        c.setFont('Helvetica-Bold', 10)        c.drawString(self.margin, y, 'Acceptance')        y -= 15        c.setFont('Helvetica', 8)        c.drawString(self.margin, y, f"Acceptance of this order via the {data['client_name']} portal constitutes agreement to all terms.")print("✅ Sales Comparison & Engagement Letter Generators ready!")

In [ ]:
# Cell 6: Additional URAR Pages (USPAP, Additional Comparables, Market Conditions)class USPAPPageGenerator:    """USPAP Identification Page"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 0.5*inch        def generate(self, c, data):        y = self.page_h - 0.5*inch                # Title        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'USPAP IDENTIFICATION')        y -= 25                # Report Type        c.setFont('Helvetica', 8)        c.drawString(self.margin, y, 'This report form is designed to report an appraisal of:')        y -= 15        c.drawString(self.margin, y, '☑ Appraisal Report (Standards Rule 2-2(a))')        y -= 12        c.drawString(self.margin, y, '☐ Restricted Appraisal Report (Standards Rule 2-2(b))')        y -= 25                # Certifications        c.setFont('Helvetica-Bold', 9)        c.drawString(self.margin, y, 'APPRAISER CERTIFICATION')        y -= 15                c.setFont('Helvetica', 7)        certifications = [            'I certify that, to the best of my knowledge and belief:',            '• The statements of fact contained in this report are true and correct.',            '• The reported analyses, opinions, and conclusions are limited only by assumptions and limiting conditions.',            '• I have no present or prospective interest in the property that is the subject of this report.',            '• I have performed no services regarding the property within the three-year period immediately preceding.',            '• I have no bias with respect to the property or parties involved.',            '• My engagement was not contingent upon developing predetermined results.',            '• My compensation is not contingent upon the value opinion reported.',            '• My analyses and conclusions were developed in conformity with USPAP.',            '• I have made a personal inspection of the property that is the subject of this report.',            '• No one provided significant assistance except as noted in this report.',        ]                for cert in certifications:            c.drawString(self.margin, y, cert[:100])            y -= 10            if len(cert) > 100:                c.drawString(self.margin + 10, y, cert[100:])                y -= 10                y -= 20                # Signature block        c.setFont('Helvetica-Bold', 8)        c.drawString(self.margin, y, 'APPRAISER:')        y -= 15        c.setFont('Helvetica', 8)        c.drawString(self.margin, y, f"Signature: ________________________________")        y -= 12        c.drawString(self.margin, y, f"Name: {data['appraiser_name']}")        y -= 12        c.drawString(self.margin, y, f"State Certification #: {data['license_number']}")        y -= 12        c.drawString(self.margin, y, f"State: {data['license_state']}")        y -= 12        c.drawString(self.margin, y, f"Expiration Date: {data['license_expiry']}")        y -= 12        c.drawString(self.margin, y, f"Date of Signature and Report: {data['report_date']}")        y -= 12        c.drawString(self.margin, y, f"Effective Date of Appraisal: {data['effective_date']}")        y -= 15        c.drawString(self.margin, y, f"Inspection of Subject: ☑ Interior and Exterior")        y -= 12        c.drawString(self.margin, y, f"Date of Inspection: {data['inspection_date']}")                # Footer        c.setFont('Helvetica', 6)        c.drawCentredString(self.page_w/2, 0.4*inch, 'Form ID14 - "TOTAL" appraisal software by a la mode, inc.')class AdditionalComparablesGenerator:    """Additional Comparables 4-6 Page"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 0.3*inch        def generate(self, c, data):        y = self.page_h - 0.25*inch                # Header        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, f"FHA/VA Case No. {data.get('fha_case_no', '')}")        c.drawRightString(self.page_w - self.margin, y, f"File # {data['file_no']}")        c.setFont('Helvetica-Bold', 8)        c.drawCentredString(self.page_w/2, y, 'Uniform Residential Appraisal Report')        y -= 15                # Title        c.setFillColor(lightgrey)        c.rect(self.margin, y-12, self.page_w - 2*self.margin, 12, fill=True, stroke=True)        c.setFillColor(black)        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin+3, y-10, 'ADDITIONAL COMPARABLES 4-6')        y -= 15                # Grid for comparables 4-6        col_w = (self.page_w - 2*self.margin) / 4        comps = data['comparables'][3:6] if len(data['comparables']) >= 6 else []                # Headers        headers = ['SUBJECT', 'COMPARABLE 4', 'COMPARABLE 5', 'COMPARABLE 6']        c.setFont('Helvetica-Bold', 6)        for i, h in enumerate(headers):            c.drawString(self.margin + i*col_w + 5, y, h)        y -= 10                # Basic info        c.setFont('Helvetica', 5)        if comps:            # Address            c.drawString(self.margin + 5, y, data['property_address'][:25])            for i, comp in enumerate(comps):                c.drawString(self.margin + (i+1)*col_w + 5, y, comp['address'][:25])            y -= 8                        # Proximity            c.drawString(self.margin + 5, y, "---")            for i, comp in enumerate(comps):                c.drawString(self.margin + (i+1)*col_w + 5, y, comp['proximity'])            y -= 10                        # Sale Price            c.setFont('Helvetica-Bold', 6)            c.drawString(self.margin + 5, y, f"${data['contract_price']}")            for i, comp in enumerate(comps):                c.drawString(self.margin + (i+1)*col_w + 5, y, f"${comp['sale_price_formatted']}")            y -= 10                        # Net Adjustment            y -= 5            c.setFont('Helvetica-Bold', 6)            c.drawString(self.margin, y, "Net Adjustment")            for i, comp in enumerate(comps):                adj = comp['net_adjustment']                sign = '+' if adj >= 0 else ''                c.drawString(self.margin + (i+1)*col_w + 5, y, f"{sign}${adj:,}")            y -= 10                        c.drawString(self.margin, y, "Adjusted Price")            for i, comp in enumerate(comps):                c.drawString(self.margin + (i+1)*col_w + 5, y, f"${comp['adjusted_price']:,}")                y -= 30                # Analysis Comments        c.setFillColor(lightgrey)        c.rect(self.margin, y-12, self.page_w - 2*self.margin, 12, fill=True, stroke=True)        c.setFillColor(black)        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin+3, y-10, 'ANALYSIS / COMMENTS')        y -= 20                c.setFont('Helvetica', 7)        comments = [            'Comparables 4-6 are additional listings and recent sales in the subject market area,',            'utilized to provide some reflection of the current market expectations. All comparables',            'were selected based on similarity to the subject property in terms of location, size,',            'quality, and condition. Adjustments were made for differences in GLA, site size, and amenities.',        ]        for comment in comments:            c.drawString(self.margin, y, comment)            y -= 10                # Footer        c.setFont('Helvetica', 5)        c.drawString(self.margin, 0.3*inch, 'Freddie Mac Form 70  March 2005')        c.drawRightString(self.page_w - self.margin, 0.3*inch, 'Fannie Mae Form 1004  March 2005')        c.drawCentredString(self.page_w/2, 0.3*inch, 'Additional Comparables')class MarketConditionsGenerator:    """Market Conditions Addendum Form 1004MC"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 0.4*inch        def generate(self, c, data):        y = self.page_h - 0.4*inch                # Title        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'Market Conditions Addendum to the Appraisal Report')        y -= 25                # Property info        c.setFont('Helvetica', 7)        c.drawString(self.margin, y, f"Property Address: {data['property_address']}")        c.drawRightString(self.page_w - self.margin, y, f"File #: {data['file_no']}")        y -= 12        c.drawString(self.margin, y, f"City: {data['city']}   State: {data['state']}   Zip: {data['zip']}")        y -= 12        c.drawString(self.margin, y, f"Borrower: {data['borrower_name']}")        y -= 25                # Instructions        c.setFont('Helvetica', 6)        inst_text = 'The purpose of this addendum is to provide the lender/client with a clear understanding of the market trends and conditions prevalent in the subject neighborhood.'        c.drawString(self.margin, y, inst_text)        y -= 20                # Inventory Analysis Table        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin, y, 'Inventory Analysis')        y -= 15                # Table headers        c.setFont('Helvetica', 6)        col_widths = [2*inch, 1.3*inch, 1.3*inch, 1.3*inch, 1*inch]        headers = ['', 'Prior 7-12 Months', 'Prior 4-6 Months', 'Current-3 Months', 'Overall Trend']        x = self.margin        for i, (h, w) in enumerate(zip(headers, col_widths)):            c.drawString(x, y, h)            x += w        y -= 12                # Table rows        inventory_data = [            ('Total # Comparable Sales', '6', '1', '2', 'Stable'),            ('Absorption Rate', '1.00', '0.33', '0.67', 'Stable'),            ('Total # Active Listings', '0', '0', '7', 'Increasing'),            ('Months of Supply', '0', '0', '10.4', 'Increasing'),        ]                for row in inventory_data:            x = self.margin            for i, (val, w) in enumerate(zip(row, col_widths)):                c.drawString(x, y, str(val))                x += w            y -= 10                y -= 15                # Median Metrics Table        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin, y, 'Median Sale & List Price, DOM')        y -= 15                c.setFont('Helvetica', 6)        x = self.margin        for h, w in zip(headers, col_widths):            c.drawString(x, y, h)            x += w        y -= 12                median_data = [            ('Median Comparable Sale Price', '$1,880,000', '$1,875,000', '$1,890,000', 'Stable'),            ('Median Sales Days on Market', '69', '64', '96', 'Increasing'),            ('Median Comparable List Price', '$0', '$0', '$2,100,000', 'Stable'),            ('Median Sale Price as % of List', '98%', '98%', '98%', 'Stable'),        ]                for row in median_data:            x = self.margin            for val, w in zip(row, col_widths):                c.drawString(x, y, str(val))                x += w            y -= 10                y -= 20                # Seller Concessions        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin, y, 'Seller Concessions:')        y -= 12        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, 'SOME SELLER CONCESSIONS WERE NOTED. HOWEVER, THE APPRAISER NOTED ONLY A SMALL')        y -= 10        c.drawString(self.margin, y, 'PERCENTAGE OF SALES INCLUDE SELLER CONCESSIONS.')        y -= 20                # Data Sources        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin, y, 'Data Sources:')        y -= 12        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, 'REALIST, MLS AND PUBLIC RECORDS')        y -= 20                # Summary        c.setFont('Helvetica-Bold', 7)        c.drawString(self.margin, y, 'Market Conditions Summary:')        y -= 12        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, data.get('market_conditions', 'Market conditions are stable with balanced supply and demand.'))                # Signature        y = 1.5*inch        c.setFont('Helvetica', 7)        c.drawString(self.margin, y, f"Signature: ________________________________")        y -= 12        c.drawString(self.margin, y, f"Appraiser Name: {data['appraiser_name']}")        y -= 12        c.drawString(self.margin, y, f"Company: {data['appraiser_company']}")                # Footer        c.setFont('Helvetica', 5)        c.drawCentredString(self.page_w/2, 0.3*inch, 'Freddie Mac Form 71  March 2009 / Fannie Mae Form 1004MC  March 2009')print("✅ Additional Pages Generators ready (USPAP, Additional Comps, Market Conditions)!")

In [ ]:
# Cell 7: Photo, Sketch & Map Pages Generatorsclass PhotoPagesGenerator:    """Generates photo pages for subject and comparables"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 0.4*inch        def generate_subject_photos(self, c, data):        """Generate subject property photo pages"""        y = self.page_h - 0.4*inch                # Header        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'SUBJECT PROPERTY PHOTOGRAPHS')        y -= 20                c.setFont('Helvetica', 7)        c.drawString(self.margin, y, f"Property Address: {data['property_address']}, {data['city']}, {data['state']} {data['zip']}")        y -= 12        c.drawString(self.margin, y, f"File #: {data['file_no']}    Borrower: {data['borrower_name']}")        y -= 25                # Photo placeholders (3 per page)        photo_h = 3.5*inch        photo_w = 5*inch        photos = [            ('Front View', 'Main facade of the subject property'),            ('Rear View', 'Rear of the subject property showing backyard'),            ('Street View', 'Street scene showing neighborhood character'),        ]                for i, (title, desc) in enumerate(photos):            # Photo box            c.rect(self.margin, y - photo_h, photo_w, photo_h, stroke=True)                        # Placeholder text            c.setFont('Helvetica', 8)            c.drawCentredString(self.margin + photo_w/2, y - photo_h/2, '[PHOTO PLACEHOLDER]')            c.setFont('Helvetica', 6)            c.drawCentredString(self.margin + photo_w/2, y - photo_h/2 - 15, title)                        # Description below            c.setFont('Helvetica-Bold', 7)            c.drawString(self.margin, y - photo_h - 15, title)            c.setFont('Helvetica', 6)            c.drawString(self.margin, y - photo_h - 25, desc)                        y -= photo_h + 40            if i < len(photos) - 1 and y < 2*inch:                # New page for additional photos                c.showPage()                y = self.page_h - 0.4*inch                # Footer        c.setFont('Helvetica', 5)        c.drawCentredString(self.page_w/2, 0.3*inch, 'Subject Property Photos')        def generate_comparable_photos(self, c, data):        """Generate comparable property photo pages"""        y = self.page_h - 0.4*inch                # Header        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'COMPARABLE SALES PHOTOGRAPHS')        y -= 20                c.setFont('Helvetica', 7)        c.drawString(self.margin, y, f"File #: {data['file_no']}")        y -= 25                # 3 comparables per page        photo_h = 2.2*inch        photo_w = 3.5*inch                comps = data['comparables'][:3]        for i, comp in enumerate(comps):            # Photo box            c.rect(self.margin, y - photo_h, photo_w, photo_h, stroke=True)                        # Placeholder            c.setFont('Helvetica', 8)            c.drawCentredString(self.margin + photo_w/2, y - photo_h/2, '[COMP PHOTO]')                        # Info beside photo            info_x = self.margin + photo_w + 0.2*inch            info_y = y - 10            c.setFont('Helvetica-Bold', 7)            c.drawString(info_x, info_y, f"Comparable {i+1}")            info_y -= 12            c.setFont('Helvetica', 6)            c.drawString(info_x, info_y, comp['address'][:35])            info_y -= 10            c.drawString(info_x, info_y, f"{comp['city']}, {comp['state']}")            info_y -= 10            c.drawString(info_x, info_y, f"Sale Price: ${comp['sale_price_formatted']}")            info_y -= 10            c.drawString(info_x, info_y, f"GLA: {comp['gla']:,} SF")            info_y -= 10            c.drawString(info_x, info_y, f"$/SF: ${comp['price_per_sf']}")            info_y -= 10            c.drawString(info_x, info_y, f"Sale Date: {comp['sale_date']}")                        y -= photo_h + 30                # Footer        c.setFont('Helvetica', 5)        c.drawCentredString(self.page_w/2, 0.3*inch, 'Comparable Sales Photos')class BuildingSketchGenerator:    """Generates building sketch and area calculations"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 0.5*inch        def generate(self, c, data):        y = self.page_h - 0.5*inch                # Header        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'BUILDING SKETCH')        y -= 20                c.setFont('Helvetica', 7)        c.drawString(self.margin, y, f"Property Address: {data['property_address']}, {data['city']}, {data['state']}")        y -= 12        c.drawString(self.margin, y, f"File #: {data['file_no']}")        y -= 30                # Sketch placeholder (large box)        sketch_h = 5*inch        sketch_w = 6*inch        sketch_x = (self.page_w - sketch_w) / 2                c.rect(sketch_x, y - sketch_h, sketch_w, sketch_h, stroke=True)                # Simple floor plan representation        c.setFont('Helvetica', 8)        c.drawCentredString(self.page_w/2, y - sketch_h/2, '[BUILDING FLOOR PLAN SKETCH]')        c.setFont('Helvetica', 6)        c.drawCentredString(self.page_w/2, y - sketch_h/2 - 15, 'Not to Scale')                # Dimensions labels        c.setFont('Helvetica', 7)        c.drawString(sketch_x + 10, y - 20, 'Main Living Area')        c.drawString(sketch_x + 10, y - 100, f"Bedrooms: {data['bedrooms']}")        c.drawString(sketch_x + 10, y - 115, f"Bathrooms: {data['bathrooms']}")        c.drawString(sketch_x + sketch_w - 100, y - 20, f"Garage: {data['garage_cars']} car(s)")                y -= sketch_h + 30                # Area Calculation Summary        c.setFont('Helvetica-Bold', 9)        c.drawString(self.margin, y, 'AREA CALCULATION SUMMARY')        y -= 15                c.setFont('Helvetica', 7)        c.drawString(self.margin, y, 'GROSS LIVING AREA (Above Grade):')        y -= 12        c.drawString(self.margin + 20, y, f"Main Level: {data['gla']} SF")        y -= 10        if int(data.get('num_stories', 1)) > 1:            c.drawString(self.margin + 20, y, f"Upper Level: {int(data['gla_raw'] * 0.4):,} SF")            y -= 10        y -= 5        c.setFont('Helvetica-Bold', 8)        c.drawString(self.margin, y, f"TOTAL GLA: {data['gla']} SF")        y -= 20                c.setFont('Helvetica', 7)        c.drawString(self.margin, y, 'NON-LIVING AREA:')        y -= 12        c.drawString(self.margin + 20, y, f"Garage: {data.get('garage_sf', 500)} SF")        y -= 10        c.drawString(self.margin + 20, y, f"Basement: {data['basement_area']} SF")                # Footer        c.setFont('Helvetica', 5)        c.drawCentredString(self.page_w/2, 0.3*inch, 'Building Sketch and Area Calculations')class MapPagesGenerator:    """Generates location, plat, and flood map pages"""        def __init__(self):        self.page_w, self.page_h = letter        self.margin = 0.4*inch        def generate_location_map(self, c, data):        """Generate location map with comparables"""        y = self.page_h - 0.4*inch                # Header        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'LOCATION MAP')        y -= 20                c.setFont('Helvetica', 7)        c.drawString(self.margin, y, f"Property Address: {data['property_address']}, {data['city']}, {data['state']}")        y -= 12        c.drawString(self.margin, y, f"File #: {data['file_no']}")        y -= 25                # Map placeholder        map_h = 6*inch        map_w = 7*inch        map_x = (self.page_w - map_w) / 2                c.rect(map_x, y - map_h, map_w, map_h, stroke=True)                # Placeholder text        c.setFont('Helvetica', 10)        c.drawCentredString(self.page_w/2, y - map_h/2, '[LOCATION MAP PLACEHOLDER]')        c.setFont('Helvetica', 7)        c.drawCentredString(self.page_w/2, y - map_h/2 - 20, 'Subject Property and Comparable Sales')                # Legend        legend_y = y - map_h/2 - 60        c.setFont('Helvetica', 6)        c.drawString(map_x + 20, legend_y, '● = Subject Property')        c.drawString(map_x + 20, legend_y - 12, '◆ = Comparable Sales')                y -= map_h + 20                # Map source        c.setFont('Helvetica', 6)        c.drawString(self.margin, y, 'Map Source: Google Maps / County GIS')                # Footer        c.setFont('Helvetica', 5)        c.drawCentredString(self.page_w/2, 0.3*inch, 'Location Map')        def generate_flood_map(self, c, data):        """Generate FEMA flood map"""        y = self.page_h - 0.4*inch                # Header        c.setFont('Helvetica-Bold', 10)        c.drawCentredString(self.page_w/2, y, 'FEMA FLOOD MAP')        y -= 20                c.setFont('Helvetica', 7)        c.drawString(self.margin, y, f"Property Address: {data['property_address']}, {data['city']}, {data['state']}")        y -= 12        c.drawString(self.margin, y, f"File #: {data['file_no']}")        y -= 12        c.drawString(self.margin, y, f"FEMA Map #: {data['flood_map_number']}    Date: {data['flood_map_date']}")        y -= 12        c.drawString(self.margin, y, f"Flood Zone: {data['flood_zone']}")        y -= 25                # Map placeholder        map_h = 6*inch        map_w = 7*inch        map_x = (self.page_w - map_w) / 2                c.rect(map_x, y - map_h, map_w, map_h, stroke=True)                # Placeholder        c.setFont('Helvetica', 10)        c.drawCentredString(self.page_w/2, y - map_h/2, '[FEMA FLOOD MAP PLACEHOLDER]')        c.setFont('Helvetica', 7)        c.drawCentredString(self.page_w/2, y - map_h/2 - 20, f"Flood Zone: {data['flood_zone']}")        c.drawCentredString(self.page_w/2, y - map_h/2 - 35, 'Subject property location marked')                y -= map_h + 20                # Flood zone info        c.setFont('Helvetica-Bold', 8)        c.drawString(self.margin, y, 'Flood Zone Information:')        y -= 12        c.setFont('Helvetica', 6)        zone_desc = {            'X': 'Area of minimal flood hazard - Outside 500-year floodplain',            'AE': 'Area subject to inundation by 1% annual chance flood',            'AH': 'Area subject to shallow flooding by 1% annual chance flood',            'A': 'Area subject to inundation by 1% annual chance flood'        }        desc = zone_desc.get(data['flood_zone'], 'See FEMA documentation for zone description')        c.drawString(self.margin, y, desc)                # Footer        c.setFont('Helvetica', 5)        c.drawCentredString(self.page_w/2, 0.3*inch, 'FEMA Flood Map')print("✅ Photo, Sketch & Map Generators ready!")

In [ ]:
# Cell 8: Enhanced PDF Assembly & Mass Generationimport osimport shutilclass ComprehensiveAppraisalAssembler:    """Assembles complete multi-page appraisal PDF with all sections"""        def __init__(self):        self.cover_gen = CoverPageGenerator()        self.urar_gen = URARPageGenerator()        self.sales_gen = SalesComparisonGenerator()        self.uspap_gen = USPAPPageGenerator()        self.add_comps_gen = AdditionalComparablesGenerator()        self.market_gen = MarketConditionsGenerator()        self.el_gen = EngagementLetterGenerator()        self.photo_gen = PhotoPagesGenerator()        self.sketch_gen = BuildingSketchGenerator()        self.map_gen = MapPagesGenerator()        def generate_appraisal(self, output_path: str, data: dict):        """Generate complete appraisal PDF with ALL pages"""        c = canvas.Canvas(output_path, pagesize=letter)                # Page 1: Cover Page        self.cover_gen.generate(c, data)        c.showPage()                # Page 2: USPAP Identification        self.uspap_gen.generate(c, data)        c.showPage()                # Page 3: URAR Page 1 (Subject, Neighborhood, Site, Improvements)        self.urar_gen.generate_page1(c, data)        c.showPage()                # Page 4: Sales Comparison (Comparables 1-3)        self.sales_gen.generate_page2(c, data)        c.showPage()                # Page 5: Additional Comparables 4-6        if len(data['comparables']) >= 6:            self.add_comps_gen.generate(c, data)            c.showPage()                # Page 6: Market Conditions Addendum        self.market_gen.generate(c, data)        c.showPage()                # Pages 7-8: Subject Property Photos        self.photo_gen.generate_subject_photos(c, data)        c.showPage()                # Page 9: Comparable Sales Photos        self.photo_gen.generate_comparable_photos(c, data)        c.showPage()                # Page 10: Building Sketch        self.sketch_gen.generate(c, data)        c.showPage()                # Page 11: Location Map        self.map_gen.generate_location_map(c, data)        c.showPage()                # Page 12: Flood Map        self.map_gen.generate_flood_map(c, data)        c.showPage()                c.save()        return output_path        def generate_engagement_letter(self, output_path: str, data: dict):        """Generate engagement letter with MATCHING data"""        c = canvas.Canvas(output_path, pagesize=letter)        self.el_gen.generate(c, data)        c.showPage()        c.save()        return output_path# Mass Generation ConfigurationNUM_PDFS = 15  # Number of matched pairs to generateOUTPUT_DIR = "generated_appraisal_documents"# Clean and create output directoriesif os.path.exists(OUTPUT_DIR):    shutil.rmtree(OUTPUT_DIR)os.makedirs(f"{OUTPUT_DIR}/appraisals")os.makedirs(f"{OUTPUT_DIR}/engagement_letters")data_gen = AppraisalDataGenerator()assembler = ComprehensiveAppraisalAssembler()appraisal_files = []engagement_files = []all_data = []print(f"\n{'='*70}")print(f"🚀 GENERATING {NUM_PDFS} COMPLETE APPRAISAL REPORT PAIRS")print(f"   Each appraisal includes:")print(f"   ✓ Cover Page")print(f"   ✓ USPAP Identification")print(f"   ✓ URAR Pages 1-2 (Subject, Neighborhood, Site, Sales Comparison)")print(f"   ✓ Additional Comparables 4-6")print(f"   ✓ Market Conditions Addendum")print(f"   ✓ Subject & Comparable Photos")print(f"   ✓ Building Sketch & Area Calculations")print(f"   ✓ Location Map & Flood Map")print(f"   ✓ MATCHING Engagement Letter")print(f"{'='*70}\n")for i in range(1, NUM_PDFS + 1):    # Generate UNIFIED data for this pair    data = data_gen.generate()    all_data.append(data)        # Create comprehensive appraisal PDF    appraisal_path = f"{OUTPUT_DIR}/appraisals/appraisal_{i:03d}.pdf"    assembler.generate_appraisal(appraisal_path, data)    appraisal_files.append(appraisal_path)        # Create MATCHING engagement letter    el_path = f"{OUTPUT_DIR}/engagement_letters/engagement_{i:03d}.pdf"    assembler.generate_engagement_letter(el_path, data)    engagement_files.append(el_path)        print(f"✅ [{i:2d}/{NUM_PDFS}] Complete Matched Pair Generated:")    print(f"   📍 {data['property_address']}, {data['city']}, {data['state']} {data['zip']}")    print(f"   👤 Borrower: {data['borrower_name']}")    print(f"   🏦 Lender: {data['lender_name']}")    print(f"   📋 Order: {data['order_number']} | File: {data['file_no']}")    print(f"   💰 Value: ${data['appraised_value']} | GLA: {data['gla']} SF")    print(f"   📄 Appraisal: ~12 pages | Engagement: 1 page")    print()print(f"{'='*70}")print(f"🎉 GENERATION COMPLETE!")print(f"   📊 Total Pairs Generated: {len(appraisal_files)}")print(f"   📁 Appraisal Reports: {len(appraisal_files)} files")print(f"   📁 Engagement Letters: {len(engagement_files)} files")print(f"   ✓ All shared data (address, borrower, lender) matches between pairs")print(f"   ✓ Each pair has completely unique data")print(f"{'='*70}\n")

In [ ]:
# Cell 7: Summary CSV & Downloadimport pandas as pdimport zipfile# Create summarysummary = [{    'Pair': i+1,    'Appraisal': f'appraisal_{i+1:03d}.pdf',    'Engagement': f'engagement_{i+1:03d}.pdf',    'Borrower': d['borrower_name'],    'Address': d['property_address'],    'City': d['city'],    'State': d['state'],    'Lender': d['lender_name'],    'Order #': d['order_number'],    'Value': f"${d['appraised_value']}"} for i, d in enumerate(all_data)]df = pd.DataFrame(summary)df.to_csv(f'{OUTPUT_DIR}/matched_pairs_summary.csv', index=False)print("📊 Matched Pairs Summary:")display(df)

In [ ]:
# Cell 8: Download All Fileszip_name = "appraisal_documents.zip"with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:    for f in appraisal_files:        zf.write(f, f"appraisals/{os.path.basename(f)}")    for f in engagement_files:        zf.write(f, f"engagement_letters/{os.path.basename(f)}")    zf.write(f'{OUTPUT_DIR}/matched_pairs_summary.csv', 'matched_pairs_summary.csv')print(f"\n📦 Created {zip_name}")print(f"   Size: {os.path.getsize(zip_name) / (1024*1024):.2f} MB")print(f"   Contains: {len(appraisal_files)} appraisals + {len(engagement_files)} engagement letters")print(f"   ⚠️ All pairs have MATCHING data (address, borrower, lender)")files.download(zip_name)print("\n⬇️ Download started!")